# Lógica difusa cuántica

La implementación de lógica difusa mediante Computación Cuántica es relativamente directa, gracias al tratamiento que se hace de la información en el modelo clásico.

In [ ]:
import pennylane as qml
import numpy as np
from itertools import combinations

In [5]:
def linzmf(x, a, b):
    if x <= a:
        return 1.0
    if x >= b:
        return 0.0
    return (b - x) / (b - a)


def linsmf(x, a, b):
    if x <= a:
        return 0.0
    if x >= b:
        return 1.0
    return (x - a) / (b - a)


def trapmf(x, a, b, c, d):
    if x <= a or x >= d:
        return 0.0
    if b <= x <= c:
        return 1.0
    if a < x < b:
        return (x - a) / (b - a)
    return (d - x) / (d - c)


def fuzzy_temp(t):
    # Dominio aproximado [0, 40]
    temp_low = linzmf(t, 10.0, 20.0)
    temp_medium = trapmf(t, 10.0, 20.0, 30.0, 40.0)
    temp_high = linsmf(t, 30.0, 40.0)
    return [temp_low, temp_medium, temp_high]

## Variables difusas

Para codificar una variable difusa en un circuito cuántico, se utiliza un registro de $q$ qubits, siendo $q = \lceil \text{log}_2 n \rceil$, donde $n$ es el número de conjuntos difusos que contiene la variable difusa a codificar. El registro se inicializa con la siguiente configuración:

$$\ket{\psi_x} = \sum\limits_{i=0}^{n-1} \sqrt{\mu_{X^i}(x)}\ket{i} + \sum\limits_{j=n}^{2^q -1} 0\ket{j}$$

donde $\mu_{X^i}(x)$ es el valor de cada una de las funciones de pertenencia de los conjuntos difusos de la variable para el dato $x$.

Comúnmente, las variables difusas se diseñan para que la suma de cada uno de sus conjuntos difusos sea 1, pero esto no tiene que ser siempre así. Para tratar con estos casos, se aplica la siguiente inicialización:

$$\ket{\psi_x} = \sum\limits_{i=0}^{n-1} \sqrt{\mu_{X^i}(x)}\ket{i} + \sum\limits_{j=n}^{2^q -2} 0\ket{j} + \sqrt{ 1 -\sum\limits_{i=0}^{n-1} \mu_{X^i}(x)}\ket{2^q - 1}$$

donde se utiliza uno de los estados libres para "codificar" la amplitud restante, y asegurar la validez del estado correspondiente. Cuando se dé el caso de que $q = log_2 n$ y haya que hacer esta regularización, será necesario que el registro de qubits sea de $q+1$ qubits.

### Ejercicio 1

Implementa una función que reciba una lista de valores de una variable difusa y una lista de qubits, y aplique el operador de inicialización correspondiente. Asegúrate de que:

* Se comprueba que la lista de valores sean un estado válido. En caso de que no, calcula lo restante.
* Se comprueba que el número de qubits es suficiente para codificar la lista de valores (y el restante en caso de que sea necesario).

Una vez implementado, aplícalo para codificar la variable temperatura que vimos en el ejemplo clásico. Puedes reutilizar aquí el mismo código.

In [ ]:
def init_quantum_fuzzy(values, wires):
    if len(values) == 0:
        raise ValueError("values no puede estar vacia")
    if any(v < 0 for v in values):
        raise ValueError("values debe contener valores no negativos")

    sum_vals = float(np.sum(values))
    if sum_vals > 1.0:
        raise ValueError("la suma de values es mayor que 1")

    # Si falta amplitud, se reserva el ultimo estado base como residual.
    needs_residual = sum_vals < 1.0
    required_states = len(values) + (1 if needs_residual else 0)
    q = int(np.ceil(np.log2(required_states))) if required_states > 1 else 1

    if len(wires) < q:
        raise ValueError("no hay suficientes qubits para codificar values")

    amplitudes = np.zeros(2**q, dtype=float)
    amplitudes[: len(values)] = np.sqrt(values)
    if needs_residual:
        amplitudes[-1] = np.sqrt(1.0 - sum_vals)

    qml.AmplitudeEmbedding(amplitudes, wires=wires[:q], normalize=False)

In [7]:
dev = qml.device("default.qubit", wires=2)


@qml.qnode(dev)
def init_quantum_fuzzy_example(values):
    init_quantum_fuzzy(values, wires=[0, 1])
    return qml.state()


t = 25
init_quantum_fuzzy_example(fuzzy_temp(t))

array([0.+0.j, 1.+0.j, 0.+0.j, 0.+0.j])

## Conectivas

Al utilizar registros de qubits para codificar las variables difusas, la implementación de las conectivas lógicas no es trivial. Se implementan mediante operadores $X$ multicontrolados, donde los qubits de control son los registros de las variables difusas, y qubit objetivo es un nuevo qubit que almacenará la amplitud correspondiente a aplicar dicha conectiva.

En primer lugar, se debe aplicar una máscara en función del estado que representa la etiqueta de la variable difusa a la que se le quiere aplicar la conectiva. Por ejemplo, si tenemos una variable difusa codificada en $q = 2$ qubits, y queremos aplicar la conectiva a la etiqueta representada por el estado $\ket{01}$, se debe aplicar la máscara $X \otimes I$.

***Nota:** Pennylane permite pasar directamente el estado con el que se quiere controlar un operador controlado. Si se utiliza esa funcionalidad, no sería necesario pasar la máscara también. Esta explicación se hace para que resulte agnóstica a la librería de programación.*

#### Ejercicio 2

Implementa una función que reciba el estado sobre el que se quiere aplicar la máscara, y devuelva la combinación de operadores $X$ e $I$ correspondientes.

In [ ]:
def mask(state: str, wires: list[int]) -> None:
    bits = "".join(state)

    if len(bits) != len(wires):
        raise ValueError("el estado no coincide con el numero de qubits")

    for bit, wire in zip(bits, wires):
        if bit == "1":
            qml.PauliX(wires=wire)

### Conjunción

Para la conectiva de conjunción, se aplica un único operador $X$ multicontrolado, con los qubits de control siendo los registros de las variables difusas (que ya han sido enmascaradas de acorde a la etiqueta correspondiente):

$$U_{conj}(\ket{x_0} \otimes ... \otimes \ket{x_n} \otimes \ket{0}) = MCX(\ket{x_0} \otimes ... \otimes \ket{x_n} \otimes \ket{0}) = \ket{x_0} \otimes ... \otimes \ket{x_n} \otimes \ket{x_1 \land ... \land x_n}$$

#### Ejercicio 3

Implementa una función que reciba un mapa de codificación y los qubits correspondientes, y aplique la conectiva de conjunción.

In [ ]:
example_map = {"x0": [0, 1, 2], "x1": [3, 4], "target": [5]}

In [ ]:
def quantum_conj(map, wires):
    # Aplica una MCX controlada por todos los registros, con el qubit objetivo al final.
    if "target" not in map or len(map["target"]) != 1:
        raise ValueError("se requiere un unico qubit objetivo en map['target']")

    control_indices = []
    for key, indices in map.items():
        if key != "target":
            control_indices.extend(indices)

    control_wires = [wires[i] for i in control_indices]
    target_wire = wires[map["target"][0]]

    qml.MultiControlledX(wires=control_wires + [target_wire])

### Disyunción

Para la conectiva de disyunción, se aplican múltiples operadores $X$ multicontrolados, cada uno controlado por un subconjunto de las etiquetas correspondientes, según las combinaciones posibles:

$$U_{disj}(\ket{x_0} \otimes \ket{x_1} \otimes \ket{0}) = MCX(\ket{x_0} \otimes \ket{x_1} \otimes \ket{0}) MCX(\ket{x_0} \otimes \ket{0}) MCX(\ket{x_1} \otimes \ket{0}) = \ket{x_0} \otimes \ket{x_1} \otimes \ket{x_0 \lor x_1}$$

#### Ejercicio 4

Implementa una función que reciba un mapa de codificación y los qubits correspondientes, y aplique la conectiva de disyunción.

In [ ]:
def quantum_disj(map, wires):
    # Aplica MCX para todas las combinaciones no vacias de registros de control.
    if "target" not in map or len(map["target"]) != 1:
        raise ValueError("se requiere un unico qubit objetivo en map['target']")

    control_groups = [indices for key, indices in map.items() if key != "target"]
    if not control_groups:
        raise ValueError("se requiere al menos un registro de control")

    target_wire = wires[map["target"][0]]

    for r in range(1, len(control_groups) + 1):
        for subset in combinations(control_groups, r):
            control_indices = [i for group in subset for i in group]
            control_wires = [wires[i] for i in control_indices]
            qml.MultiControlledX(wires=control_wires + [target_wire])

#### Ejercicio 5

Define una nueva variable difusa ```humedad``` de acuerdo al siguiente criterio:

$\mathcal{D}_{hum} = [0, 100]$

$hum_{low}(t) = linzmf_{[10, 20]}(t)$

$hum_{medium}(t) = trapmf_{[10, 20, 60, 80]}(t)$

$hum_{high}(t) = linzmf_{[60, 80]}(t)$

Utiliza las variables difusas ```temperatura``` y ```humedad``` para definir las siguientes conectivas:

* $temp_{low}\ OR\ hum_{medium}$

* $temp_{high}\ AND\ hum_{high}$

Implementa cada conectiva en su respectivo circuito cuántico, y muestra los resultados obtenidos para $temp = 25$ y $hum = 70$.

In [ ]:
def fuzzy_hum(h):
    # Funciones de pertenencia para humedad.
    hum_low = linzmf(h, 10.0, 20.0)
    hum_medium = trapmf(h, 10.0, 20.0, 60.0, 80.0)
    hum_high = linzmf(h, 60.0, 80.0)
    return [hum_low, hum_medium, hum_high]


# Asignacion de qubits: dos para cada variable y uno objetivo.
temp_wires = [0, 1]
hum_wires = [2, 3]
target_wire = 4
labels = {"low": [0, 0], "medium": [0, 1], "high": [1, 0]}


dev_or = qml.device("default.qubit", wires=5)


@qml.qnode(dev_or)
def temp_low_or_hum_medium(temp, hum):
    # Prepara los registros difusos.
    init_quantum_fuzzy(fuzzy_temp(temp), temp_wires)
    init_quantum_fuzzy(fuzzy_hum(hum), hum_wires)

    # Implementa OR como combinacion de controles (x, y, x+y).
    qml.MultiControlledX(
        wires=temp_wires + [target_wire],
        control_values=labels["low"],
    )
    qml.MultiControlledX(
        wires=hum_wires + [target_wire],
        control_values=labels["medium"],
    )
    qml.MultiControlledX(
        wires=temp_wires + hum_wires + [target_wire],
        control_values=labels["low"] + labels["medium"],
    )

    return qml.probs(wires=[target_wire])


dev_and = qml.device("default.qubit", wires=5)


@qml.qnode(dev_and)
def temp_high_and_hum_high(temp, hum):
    # Prepara los registros difusos.
    init_quantum_fuzzy(fuzzy_temp(temp), temp_wires)
    init_quantum_fuzzy(fuzzy_hum(hum), hum_wires)

    # AND como un unico control conjunto.
    qml.MultiControlledX(
        wires=temp_wires + hum_wires + [target_wire],
        control_values=labels["high"] + labels["high"],
    )

    return qml.probs(wires=[target_wire])


temp_val = 25
hum_val = 70
print("temp_low OR hum_medium:", temp_low_or_hum_medium(temp_val, hum_val))
print("temp_high AND hum_high:", temp_high_and_hum_high(temp_val, hum_val))